## Predict lander horizon using past

Suppose that we have data $a_0, \ldots, a_{n - 1}$ for some $n \in \mathbb{Z}_{> 0}$, all uniformly sampled in time.
We adopt a wierd convention for computational purposes: $a_0$ is the most recent time, and $a_{n - 1}$ is the time furthest in the past.
Say that $a_k$ was sampled at time $t_k = -k \, h$, where $1 / h$ is the sampling frequency.
Let $N$ be the number of times that we want to sample in the future.
We want to find a polynomial, say of degree $3$, such that $p(t) \in \mathcal{P}^3$ satsifies $p(0) = a_0$ and minimizes the functional
\begin{equation*}
\min_{K, r_0, r_1} \frac{\alpha_0}{n} \sum_{k = 0}^{n - 1} [e^{\beta \, t_k} |p(t_k) - a_k|^2] + \alpha_1 \, |p(r_0) - p(r_1)|^2 + \alpha_2 \, (|p(-(n - 1) \, h)|^2 + |p((N - 1) \, h)|^2)
\end{equation*}
for some (tuned) parameters $\alpha_k$ and $\beta$, and where we adopt the parameterization
\begin{equation*}
  p'(t) = K \, (t - r_0) \, (t - r_1) \quad\implies\quad p(t) = \frac{K}{3} \, t^3 - \frac{K}{2} \, (r_0 + r_1) \, t^2 + K \, r_0 \, r_1 \, t + a_0.
\end{equation*}
Namely, we want to do some exponentially weighted least squares for matching past data, but we regularize the cubic polynomial so that it isn't too large, particularly over our data and future sampling times.

In [ ]:
%matplotlib ipympl

In [ ]:
from __future__ import annotations

import dataclasses
import functools
import typing as tp
import pandas as pd
import numpy as np
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import scipy.interpolate as sci_interp
import scipy.optimize as sci_opt
import scipy.signal as sci_sig
import control as ct

import exp_mpc.stewart_min.siso as siso
import exp_mpc.stewart_min.mpc_spec as mpc_spec
import lbfgs.lbfgs as lbfgs

jax.config.update("jax_enable_x64", True)

In [ ]:
def load_clean_references(file_path: str) -> tuple[jax.Array, jax.Array]:
    data = np.array(pd.read_hdf(file_path))
    return data[:, 1:4], data[:, 4:]

# file_path = "/Users/jozbee/work/eng/comp/data/clean_specific-forces-lander_motion_redes_auto.hdf"
file_path = "/Users/jozbee/work/eng/comp/data/clean_specific-forces-lander_motion_redes_manual.hdf"
acc_ref, omega_ref = load_clean_references(file_path)

acc_ref = acc_ref[::2]
omega_ref = omega_ref[::2]

# acc_ref = jnp.clip(acc_ref, min=-1.0, max=1.0)
omega_ref = jnp.clip(omega_ref, min=-0.8, max=0.8)
s = ct.tf("s") / (2 * np.pi * 0.5)
butter = siso.DiscreteSISO.cont2discrete(1 / (1 + 2 * s + 2 * s**2 + s**3), dt=mpc_spec.dt)
butter_int = jax.jit(jax.vmap(lambda u: siso.lti_int(butter.E0, butter.E1, butter.C, butter.D, jnp.zeros(3), u)[1], in_axes=1))
acc_ref = butter_int(acc_ref).T
omega_ref = butter_int(omega_ref).T

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 5))
ax.plot(acc_ref[:, 2])
ax.grid()

In [ ]:
@jax.jit
def poly(a0, r0, r1, K, t):
    return (K / 3) * t**3 - K * (r0 + r1) * t**2 + K * r0 * r1 * t + a0

def poly_cost(poly_params, cost_params, a, h, N):
    r0, r1, K = poly_params
    alpha0, alpha1, alpha2, beta = cost_params

    p = functools.partial(poly, a[0], r0, r1, K)
    t = -jnp.arange(0, a.size, dtype=float) * h
    
    term0 = alpha0 * jnp.mean(jnp.exp(beta * t) * jnp.square(p(t) - a))
    term1 = alpha1 * jnp.square(p(r0) - p(r1))
    term2 = alpha2 * (jnp.square(p(t[-1])) + jnp.square(p((N - 1) * h)))
    return (term0 + term1 + term2) * 1e5

poly_cost_and_grad = jax.jit(jax.value_and_grad(poly_cost))

def poly_pred_sci_opt(data, cost_params, h, N, output_past=False):
    a = jnp.flip(data)
    min_fun = functools.partial(
        poly_cost_and_grad,
        cost_params=cost_params,
        a=a,
        h=h,
        N=N,
    )
    res = sci_opt.minimize(
        fun=min_fun,
        # x0=np.array([-h, h, 1.0]),
        x0=np.zeros(3),
        method="L-BFGS-B",
        jac=True,
        options={
            # "maxiter": 16,
            # "maxls": 4,
        #     "ftol": 1e-12,
        #     "gtol": 1e-12,
        }
    )
    print(res)
    p = functools.partial(poly, a[0], res.x[0], res.x[1], res.x[2])
    if output_past:
        t = jnp.arange(-a.size + 1, N, dtype=float) * h
    else:
        t = jnp.arange(0, N, dtype=float) * h
    return p(t)

@functools.partial(jax.jit, static_argnames=["h", "N", "output_past"])
def poly_pred_jax(data, cost_params, h, N, output_past=False):
    a = jnp.flip(data)

    def lbfgs_cost_grad(_, poly_params):
        return poly_cost_and_grad(poly_params, cost_params, a, h, N)

    opt_params = lbfgs.OptParamsLBFGS(
        fun=lbfgs_cost_grad,
        max_iter=16,
        max_ls=4,
        tol=1e-5,
        c1=1e-4,
        c2=0.9,
    )
    res = lbfgs.lbfgs(
        opt_params=opt_params,
        # x0=jnp.array([-h, h, 1.0]),
        x0=jnp.zeros(3),
        fun_params=tuple(),
        unroll=True,
    )
    # jax.debug.print("res = {res}", res=res)
    p = functools.partial(poly, a[0], res[0][0], res[0][1], res[0][2])
    if output_past:
        t = jnp.arange(-a.size + 1, N, dtype=float) * h
    else:
        t = jnp.arange(0, N, dtype=float) * h
    return p(t)

In [ ]:
# assert False
data = acc_ref[:, 0]
idx = 6050
pred = poly_pred_sci_opt(data[idx: idx + 50], jnp.array([1.0, 5e-4, 5e-4, 0.0]), 0.01, 200, True)
# pred = poly_pred_jax(data[idx: idx + 50], jnp.array([1.0, 5e-4, 5e-4, 0.0]), 0.01, 200, True)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(jnp.arange(idx, idx + 250), data[idx: idx + 250], label="data")
ax.plot(jnp.arange(idx + 0, idx + 250 - 1), pred, label="pred")
ax.grid()
ax.legend()

In [ ]:
%timeit -n 10 -r 1 poly_pred_jax(data[idx: idx + 50], jnp.array([1.0, 5e-4, 5e-4, 0.0]), 0.01, 200, True)

In [ ]:
batch_poly_pred = jax.jit(jax.vmap(poly_pred_jax, in_axes=[0, None, None, None, None]), static_argnums=[2, 3, 4])
# batch_poly_pred = jax.vmap(poly_pred_jax, in_axes=[0, None, None, None, None])

In [ ]:
dt = 0.01
n = 50
N = 200
data_repeat = jnp.fromfunction(
    lambda i, j: data[i + j],
    shape=(data.size - n - N, n),
    dtype=int,
)
cost_params = jnp.array([1.0, 5e-4, 5e-4, 0.0])

In [ ]:
pred = batch_poly_pred(data_repeat, cost_params, dt, N, False)

In [ ]:
actual = jnp.fromfunction(
    lambda i, j: data[i + n - 1 + j],
    shape=(data.size - n - N, N),
    dtype=int,
)
jnp.mean(jnp.mean(jnp.square(pred - actual), axis=1))

In [ ]:
jnp.argmax(jnp.square(pred)) // 200